# CafChem GPT — Fine-tune on Tyrosinase (Colab GPU)

Fine-tunes the PyTorch foundation (`data/GPT_ZN305_pytorch.pt`) on
`Tyrosinase1239_IC50.csv`: a frozen phase (foundation blocks frozen, 2 new
blocks + head trained) then an unfrozen phase (everything trainable). Saves
`GPT_Tyrosinase_finetuned.pt` for download.

**Skip-frozen (default).** The frozen checkpoint
`GPT_Tyrosinase_finetuned_frozen.pt` is committed in the repo, so the clone
already has it. With `SKIP_FROZEN = True` (default) this notebook loads it and
runs ONLY the unfrozen phase on the GPU — taking advantage of the frozen work
already done locally, no re-running, no upload. Set `SKIP_FROZEN = False` to
re-run the frozen phase from the foundation instead (e.g. if the frozen
checkpoint is missing — the cell then prompts you to upload it).

Everything else (foundation `.pt`, Tyrosinase CSV, vocab) is in the repo too.
Cells run top-to-bottom.

## 1. Clone the repo


In [ ]:
import os, shutil
REPO_DIR = 'SMILES_GPT'
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone https://github.com/MauricioCafiero/SMILES_GPT.git {REPO_DIR}
os.chdir(REPO_DIR)
print('Cwd:', os.getcwd())
print('Files:', os.listdir('.'))

## 2. Install dependencies

Colab ships torch (CUDA), pandas, numpy. Only rdkit is needed.


In [ ]:
!pip install -q rdkit
import rdkit
print('rdkit', rdkit.__version__)

## 3. Verify GPU + foundation + frozen checkpoint


In [ ]:
import torch, os, sys
sys.path.insert(0, 'code')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('WARNING: no GPU — Runtime > Change runtime type > T4 GPU')

ckpt = 'data/GPT_ZN305_pytorch.pt'
c = torch.load(ckpt, map_location='cpu', weights_only=False)
print(f'\nFoundation {ckpt}: config {c["config"]}')
print('block0 q_proj:', c['state_dict']['blocks.0.q_proj.weight'].shape, '(expect [512,256] = key_dim=256)')

frz = 'GPT_Tyrosinase_finetuned_frozen.pt'
print(f'\nFrozen checkpoint {frz}: ' + ('present (SKIP_FROZEN will use it)' if os.path.exists(frz) else 'MISSING — set SKIP_FROZEN=False or upload it'))

## 4. Fine-tune

`train_gpt` auto-enables bf16 autocast on CUDA (big speedup, no accuracy cost).


In [ ]:
#@title Fine-tune config
SKIP_FROZEN    = True    #@param {type:'boolean'}
FROZEN_EPOCHS  = 25      #@param {type:'integer'}
UNFROZEN_EPOCHS= 25      #@param {type:'integer'}
BATCH_SIZE     = 512     #@param {type:'integer'}
LR             = 1e-3    #@param {type:'number'}

import os, sys, numpy as np
sys.path.insert(0, 'code')
from CafChemGPT import (test_vocab, trim_vocab, make_datasets, make_finetune_gpt,
                        unfreeze_gpt, train_gpt, save_gpt, load_gpt, get_device)

DEVICE = get_device()
FINETUNE_FILE = 'GPT_Tyrosinase_finetuned'   # .pt appended by save_gpt

# --- data prep (same for both paths) ---
DATA = 'Tyrosinase1239_IC50.csv'
novel = test_vocab(DATA, 'SMILES')
trim_vocab(DATA, novel)
fx, fy, VOCAB_SIZE, tokenizer, max_length = make_datasets('Tyrosinase1239_IC50_trimmed.csv', 'SMILES')
print(f'fx {fx.shape} | fy {fy.shape} | VOCAB {VOCAB_SIZE} | max_len {max_length} | device {DEVICE}')

if SKIP_FROZEN:
    # Take advantage of the frozen checkpoint already in the repo. Load it and
    # run only the unfrozen phase. (Upload fallback if it's somehow missing.)
    frozen_path = FINETUNE_FILE + '_frozen.pt'
    if not os.path.exists(frozen_path):
        print(f'{frozen_path} not found in clone — upload it now.')
        from google.colab import files
        uploaded = files.upload()
        assert frozen_path in uploaded, f'expected {frozen_path}; got {list(uploaded)}'
    # Frozen model is 4 blocks (2 foundation + 2 new). load_gpt rebuilds all-trainable.
    gpt_ft = load_gpt(FINETUNE_FILE + '_frozen', total_layers=4, max_length=max_length, VOCAB_SIZE=VOCAB_SIZE)
    gpt_ft = unfreeze_gpt(gpt_ft)   # explicit; load_gpt already leaves all trainable
else:
    # Frozen phase: foundation frozen, train the 2 new blocks + head.
    gpt_ft = make_finetune_gpt(2, freeze_old_layers=True)
    gpt_ft = train_gpt(gpt_ft, fx, fy, epochs=FROZEN_EPOCHS, batch_size=BATCH_SIZE, lr=LR)
    save_gpt(gpt_ft, FINETUNE_FILE + '_frozen')
    print(f'Frozen-only model saved -> {FINETUNE_FILE}_frozen.pt')
    gpt_ft = unfreeze_gpt(gpt_ft)

# --- unfrozen phase (all 2.7M params trainable) ---
gpt_ft = train_gpt(gpt_ft, fx, fy, epochs=UNFROZEN_EPOCHS, batch_size=BATCH_SIZE, lr=LR)
save_gpt(gpt_ft, FINETUNE_FILE)
print(f'\nFine-tuned model saved -> {FINETUNE_FILE}.pt')

## 5. Download the fine-tuned `.pt` back to your Mac


In [ ]:
from google.colab import files
import os
p = 'GPT_Tyrosinase_finetuned.pt'
print(f'Downloading {p} ({round(os.path.getsize(p)/1e6,1)} MB) ...')
files.download(p)

## 6. (Optional) Generation sanity check

Generate a few molecules from the fine-tuned model to confirm valid SMILES.


In [ ]:
import sys; sys.path.insert(0, 'code')
from CafChemGPT import load_gpt, make_prompts, gen_mols
from smiles_tokenizer import SmilesTokenizer
from IPython.display import display
from rdkit import Chem

tok = SmilesTokenizer(vocab_file='data/vocab_305K.txt')
VOCAB = tok.vocab_size()
model = load_gpt('GPT_Tyrosinase_finetuned', 4, 166, VOCAB)

prompts = make_prompts(8, 2)
pic, smiles = gen_mols(prompts, use_ramp=False, model=model, tokenizer=tok,
                       TEMP=0.7, VOCAB_SIZE=VOCAB)
valid = [s for s in smiles if Chem.MolFromSmiles(s) is not None]
print(f'{len(valid)}/{len(smiles)} valid SMILES. Samples:', smiles[:5])
if hasattr(pic, 'save'):
    pic.save('finetune_gen.png')
    from IPython.display import Image
    display(Image('finetune_gen.png'))
else:
    display(pic)